<a href="https://colab.research.google.com/github/olihile84-tech/exercise-syntax-variables-and-numbers/blob/main/test_2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import pandas as pd

trades = {
    'pair':    ['EURUSD', 'GBPJPY', 'EURUSD', 'XAUUSD', 'GBPJPY'],
    'session': ['London', 'Tokyo', 'New York', 'London', 'New York'],
    'entry':   [1.1000, 185.00, 1.0950, 1920.00, 184.50],
    'exit':    [1.1050, 184.50, 1.0900, 1935.00, 185.20],
    'sl':      [1.0980, 185.20, 1.0970, 1915.00, 184.20],
    'result':  ['win', 'loss', 'loss', 'win', 'win']
}

journal = pd.DataFrame(trades)

In [41]:

ACCOUNT_BALANCE = 10000
RISK_PERCENT = 0.25
PIP_VALUE = 10


trades = {
    'pair':    ['EURUSD', 'GBPJPY', 'EURUSD', 'XAUUSD', 'GBPJPY'],
    'session': ['London', 'Tokyo', 'New York', 'London', 'New York'],
    'entry':   [1.1000, 185.00, 1.0950, 1920.00, 184.50],
    'exit':    [1.1050, 184.50, 1.0900, 1935.00, 185.20],
    'sl':      [1.0980, 185.20, 1.0970, 1915.00, 184.20],
    'result':  ['win', 'loss', 'loss', 'win', 'win']
}

journal = pd.DataFrame(trades)

In [42]:
def calculate_rr(row):
  risk = abs(row['entry'] - row['sl'])
  reward = abs (row['exit'] - row['entry'])
  return 0 if risk == 0 else round(reward/risk,2)

journal['rr'] = journal.apply(calculate_rr, axis=1)


In [43]:
def calculate_pip(row):
  diff = row['exit'] - row['entry']
  if 'JPY' in row['pair']:
    return round(diff*100, 2)
  elif 'XAU' in row['pair']:
    return round(diff*10,2)
  else:
    return round(diff*10000,2)

journal['pips'] =  journal.apply(calculate_pip, axis=1)

In [44]:

def calculate_pnl(row):
  cash_at_risk = ACCOUNT_BALANCE*(RISK_PERCENT/100)
  if row['result'] == 'win':
    return round(cash_at_risk*row['rr'],2)
  else:
    return round(-cash_at_risk, 2)

journal['pnl'] = journal.apply(calculate_pnl, axis=1)

In [45]:
def calculate_position_size(row):
  cash_at_risk = ACCOUNT_BALANCE * (RISK_PERCENT / 100)
  sl_pips = abs(row['entry'] - row['sl'])

  if 'JPY' in row['pair']:
    sl_pips = sl_pips * 100
  elif 'XAU' in row['pair']:
    sl_pips = sl_pips * 10
  else:
    sl_pips = sl_pips * 10000

  lot_size = round(cash_at_risk / (sl_pips* PIP_VALUE), 2)



In [46]:
session_stats = journal.groupby('session')['result'].apply(
    lambda x: (x=='win').sum()/ len(x) * 100
).round(2)
print('Win rate by session:')
print(session_stats)
print()

Win rate by session:
session
London      100.0
New York     50.0
Tokyo         0.0
Name: result, dtype: float64



In [47]:
best_pair = journal.groupby('pair')['rr'].mean().sort_values(ascending=False)
print("Average RR per pair:")
print(best_pair)
print()

Average RR per pair:
pair
XAUUSD    3.000
EURUSD    2.500
GBPJPY    2.415
Name: rr, dtype: float64



In [48]:
total_trades = len(journal)
wins = len(journal[journal['result']=='win'])
win_rate = (wins/total_trades)*100
avg_rr = journal['rr'].mean()
total_pnl = journal['pnl'].sum()

print(f'Total Trades: {total_trades}')
print(f'Wins: {wins}')
print(f'Win Rate: {win_rate:.2f}%')
print(f'Average RR: {round(avg_rr,2)}')
print()

def calculate_position_size(row):
  cash_at_risk = ACCOUNT_BALANCE * (RISK_PERCENT/100)
  sl_pips = abs(row['entry'] - row['sl'])

  if 'JPY' in row['pair']:
    sl_pips = sl_pips * 100
  elif 'XAU' in row['pair']:
    sl_pips = sl_pips * 10
  else:
    sl_pips = sl_pips * 10000

  lot_size = round(cash_at_risk / (sl_pips * PIP_VALUE), 2)
  return lot_size


journal['lot_size'] = journal.apply(calculate_position_size, axis=1)
journal['cash_at_risk'] = round(ACCOUNT_BALANCE*(RISK_PERCENT/100), 2)


Total Trades: 5
Wins: 3
Win Rate: 60.00%
Average RR: 2.57



In [49]:
print("RR values:")
print(journal['rr'])
print()
print("Result values:")
print(journal['result'])
print()
print("PNL calculations check")
for i, row in journal.iterrows():
  cash_at_risk = ACCOUNT_BALANCE * (RISK_PERCENT/100)
  if row['result']== 'win':
    expected = round(cash_at_risk * row['rr'], 2)
  else:
    expected = round(-cash_at_risk, 2)
  print(f"Row{i}:result={row['result']}, rr={row['rr']}, expected_pnl={expected}, actual_pnl={row['pnl']}")

RR values:
0    2.50
1    2.50
2    2.50
3    3.00
4    2.33
Name: rr, dtype: float64

Result values:
0     win
1    loss
2    loss
3     win
4     win
Name: result, dtype: object

PNL calculations check
Row0:result=win, rr=2.5, expected_pnl=62.5, actual_pnl=62.5
Row1:result=loss, rr=2.5, expected_pnl=-25.0, actual_pnl=-25.0
Row2:result=loss, rr=2.5, expected_pnl=-25.0, actual_pnl=-25.0
Row3:result=win, rr=3.0, expected_pnl=75.0, actual_pnl=75.0
Row4:result=win, rr=2.33, expected_pnl=58.25, actual_pnl=58.25


In [50]:
print(journal[['pair','session','result','rr','pips','lot_size','cash_at_risk','pnl']])
print()
print(f'Total P&L: ${total_pnl}')
print(f'Starting Balance: ${ACCOUNT_BALANCE}')
print(f'Final Balance: ${ACCOUNT_BALANCE+total_pnl}')

     pair   session result    rr   pips  lot_size  cash_at_risk    pnl
0  EURUSD    London    win  2.50   50.0      0.12          25.0  62.50
1  GBPJPY     Tokyo   loss  2.50  -50.0      0.13          25.0 -25.00
2  EURUSD  New York   loss  2.50  -50.0      0.12          25.0 -25.00
3  XAUUSD    London    win  3.00  150.0      0.05          25.0  75.00
4  GBPJPY  New York    win  2.33   70.0      0.08          25.0  58.25

Total P&L: $145.75
Starting Balance: $10000
Final Balance: $10145.75


In [51]:
journal.to_csv('trading_journal.csv', index=False)
print("\nJournal saved to trading_journal.csv!")


Journal saved to trading_journal.csv!
